# 串扰仿真 v02 —— 短焦 16 激光器编码串扰检测（基于 Elephant 时序，只看 Group A）

> 理想回波模型（δ 函数、100% 探测率），不考虑蒙卡与波形展宽。
> 时序图从 `Elephant 时序表.xlsx` 长焦 tab 的 Group A 读取。

## 核心概念
- **Kick（触发节拍）**：Group A 的 16 个 kick，每个 kick 有 16 个激光器发光。
- **编码**：`tx_trig_dly`（1ns 步长）+ `tdelay`（1/12ns 步长）。
- **TOF 窗**：固定 2000ns（0.2us）。
- **串扰鬼影**：激光器 A 的回波落入激光器 B 的 TOF 窗，B 按自己的 `echo_time - B_fire_time` 计算距离。

> 缩写：TOF（Time of Flight，飞行时间）。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import openpyxl

for _f in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC"]:
    try:
        matplotlib.rcParams["font.sans-serif"] = [_f]; break
    except Exception:
        continue
matplotlib.rcParams["axes.unicode_minus"] = False

C_LIGHT = 2.99792458e8
NS = 1e-9
print("模块导入完成。")


In [ ]:
# ============================================================================
# 从 Excel 读取 Group A 时序（只看 A 组）
# ============================================================================
EXCEL_FILE = "Elephant 时序表.xlsx"
wb = openpyxl.load_workbook(EXCEL_FILE)
ws = wb[wb.sheetnames[0]]  # 第一个 tab = 长焦

# ---- 解析行 1-6 的标头 ----
# 第 4 行: tdelay (1/12ns 步长)
tdelay_A = {}
for c in range(8, 24):  # Group A 列 8-23
    v = ws.cell(4, c).value
    if v is not None: tdelay_A[c - 8] = v

# 第 5 行: gate (ns)
gate_A = {}
for c in range(8, 24):
    v = ws.cell(5, c).value
    if v is not None: gate_A[c - 8] = v

# ---- 解析激光器数据行 (7~85) ----
# 每行: C1=DSP cluster, C2=HDC, C3=TX_TRG, C4=RX_TRG, C5=CHG, C6=LASER
# 然后列 8-23 (组A kick 0~15) 标记是否发光
# 非空 = 发激光; 值(0/50/...)就是 tx_trig_dly (1ns 步长)

laser_data = []
for r in range(7, 86):
    laser_id = ws.cell(r, 6).value
    if laser_id is None: continue
    try: laser_id = int(laser_id)
    except (ValueError, TypeError): continue
    fires = []
    for c in range(8, 24):  # Group A 列
        v = ws.cell(r, c).value
        if v is not None:
            fires.append((c - 8, int(v)))  # (kick_idx, tx_trig_dly)
    laser_data.append({
        "laser_id": laser_id,
        "tx_trg": str(ws.cell(r, 3).value or ""),
        "fires": fires,
    })

N_LASERS = len(laser_data)
print(f"从 {EXCEL_FILE} 读取 Group A: {N_LASERS} 个激光器")
print(f"  {'Laser':>6} {'TX_TRG':>6} {'发光次数':>8} {'Kick(tx_trig_dly)'}")
print(f"  {'-'*6} {'-'*6} {'-'*8} {'-'*30}")
for ld in laser_data:
    ks = ",".join(f"K{k}={d}" for (k, d) in ld["fires"])
    print(f"  {ld['laser_id']:>6d} {ld['tx_trg']:>6} {len(ld['fires']):>8d} {ks}")
print(f"\n总激光器: {N_LASERS}, 总发光事件: {sum(len(ld['fires']) for ld in laser_data)}")


In [ ]:
# ============================================================================
# 计算发光时刻（只看 Group A）
# ============================================================================
# 每个激光器的总偏移 = tx_trig_dly*1ns + tdelay[kick_idx]/12ns + fpga_jitter*8ns
# KICK_SPACING = 2.2us（两个 tick 间隔），TOF = 2000ns（固定）

KICK_SPACING = 2.2e-6        # 相邻 kick 间隔 [s]（2.2us）
KICK_BASE_START = 0.0         # 第一个 kick 的起始时刻 [s]

# FPGA 抖动暂设 0（可调）
FPGA_JITTER = np.zeros(N_LASERS + 1)   # {laser_id: 8ns步长数}

def laser_fire_offset(kick_idx, tx_trig_dly):
    """该激光器在该 kick 中的总偏移 [s] = tx_trig_dly·1ns + tdelay[kick_idx]/12ns"""
    td = tdelay_A.get(kick_idx, 0) * (1e-9 / 12.0)
    return tx_trig_dly * 1e-9 + td

def kick_base_time(kick_idx):
    return KICK_BASE_START + kick_idx * KICK_SPACING

def kick_tof_us(kick_idx):
    return 2.0  # 固定 TOF = 2000ns（0.2us）

# ---- 计算所有发光事件 ----
firing_events = []  # (laser_id, kick_idx, tx_trig_dly, fire_time[s])
for ld in laser_data:
    lid = ld["laser_id"]
    for (kidx, tr) in ld["fires"]:
        base = kick_base_time(kidx)
        off = laser_fire_offset(kidx, tr)
        t_fire = base + off
        firing_events.append((lid, kidx, tr, t_fire))

firing_events.sort(key=lambda x: x[3])
print(f"共 {len(firing_events)} 次发光事件 (按时间升序):")
print(f"  {'Laser':>6} {'Kick':>5} {'tx_trig_dly':>12} {'FireTime[ns]':>15}")
for lid, kidx, tr, tf in firing_events:
    print(f"  {lid:>6d} {kidx:>5d} {tr:>12d} {tf/NS:>15.3f}")

In [ ]:
# ============================================================================
# 16 通道回波接收：每个激光器收到哪些回波
# ============================================================================
# 对给定目标距离 D，遍历所有发光事件，检查被哪些激光器接收。
# 接收判断：激光器 B 在自己的 TOF 窗 [fire_time, fire_time + 2us] 内收到回波
# TOF 窗 = [B 的发光时刻, B 的发光时刻 + 2us]

def detect_echoes_for_target(D, verbose=False):
    """对距离 D 处的目标，返回所有回波记录。
    每项: ( emitter_laser, kick_idx, fire_time,
            detector_laser, det_kick_idx, det_fire_time,
            echo_time, calc_distance )"""
    t_tof = 2.0 * D / C_LIGHT
    results = []
    for (emit_lid, kidx_e, tr_e, t_fire_e) in firing_events:
        t_echo = t_fire_e + t_tof
        for ld in laser_data:
            det_lid = ld["laser_id"]
            det_fire = 0.0; det_kidx = -1
            for (dl, dk, dtr, dtf) in firing_events:
                if dl == det_lid and dtf <= t_echo:
                    det_fire = dtf; det_kidx = dk
            tof_win_end = det_fire + 2.0e-6  # 固定 TOF = 2us
            if det_fire > 0 and det_fire <= t_echo <= tof_win_end:
                calc_dist = (t_echo - det_fire) * C_LIGHT / 2.0
                results.append((emit_lid, kidx_e, t_fire_e,
                                det_lid, det_kidx, det_fire,
                                t_echo, calc_dist))
    return results

# ---- 测试：一个目标 ----
D_test = 100.0
res = detect_echoes_for_target(D_test)
print(f"测试目标: D={D_test:.0f}m")
print(f"  共 {len(res)} 条回波记录")
emit_set = set(r[0] for r in res)
det_set  = set(r[4] for r in res)
print(f"  发射激光器数: {len(emit_set)}")
print(f"  接收激光器数: {len(det_set)}")
if len(res) > 0:
    print(f"  首条: 发射 Laser{res[0][0]} → 接收 Laser{res[0][3]}, 计算距离={res[0][7]:.2f}m")

In [ ]:
# ============================================================================
# 绘图 1：16 通道回波接收图 —— 每个通道收到的信号位置
# ============================================================================
# 横轴 = 时间 [ns]，纵轴 = 16 个激光器通道
# 用色块/横线表示每个通道在不同时刻收到回波
# 颜色 = 发射激光器（区分不同光源）
# 每条回波线从该激光器的发光时刻到回波时刻，计算为 (echo_time - fire_time)

D_SHOW = 150.0       # 示例距离（可调）
res_show = detect_echoes_for_target(D_SHOW)

fig, ax = plt.subplots(figsize=(16, 6))

# 为每个发射激光器分配颜色
emit_ids = sorted(set(r[0] for r in res_show))
emit_colors = {eid: plt.cm.tab20(i / max(1, len(emit_ids) - 1))
               for i, eid in enumerate(emit_ids)}

# 画每个接收通道的"时间线"
for det_lid in range(1, N_LASERS + 1):
    # 该通道收到的回波
    echoes = [r for r in res_show if r[3] == det_lid]
    if not echoes:
        # 空通道画一条浅灰线
        ax.axhline(det_lid - 0.33, det_lid + 0.33, color="0.9", lw=0.5)
        continue
    for (emit_lid, kidx_e, _t_fire_e, _det_lid, _det_kidx, det_fire, t_echo, calc_d) in echoes:
        color = emit_colors.get(emit_lid, "gray")
        # 画水平线段：从该探测器发光时刻到回波时刻（相对时间）
        ax.hlines(det_lid, det_fire, t_echo, colors=color, lw=2.5, alpha=0.8)
        # 在回波时刻画圆点
        ax.plot(t_echo, det_lid, "o", color=color, ms=6, zorder=5)
        # 标明该信号来自哪个激光器（在图例或附近标注）
        if det_lid == 1:  # 只在第一个通道标一次
            ax.text(t_echo, det_lid + 0.5, f"来自 Laser{emit_lid}",
                    fontsize=6, color=color, ha="left", va="bottom")

    # 横轴改为 ns 显示（原坐标是秒，此处转换为 ns 显示）
    import matplotlib.ticker as ticker
    def _sec_to_ns(x, pos): return f"{x*1e9:.0f}"
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(_sec_to_ns))

# 标注
ax.set_xlabel("时间 t [ns]")
ax.set_ylabel("激光器通道 (接收)")
ax.set_yticks(range(1, N_LASERS + 1))
ax.set_ylim(0.5, N_LASERS + 0.5)
ax.set_title(f"16 通道回波接收图：D={D_SHOW:.0f}m"
             f"（颜色=发射激光器，横线=发光→回波时间差，圆点=回波时刻）")
ax.grid(alpha=0.3, axis="x")

# 图例（发射器颜色）
for eid in emit_ids:
    ax.plot([], [], "-", color=emit_colors[eid], lw=2, label=f"Laser {eid} 发射")
ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout()
plt.savefig("crosstalk_v02_echo_16ch.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"回波图: D={D_SHOW:.0f}m, {len(res_show)} 条回波, "
      f"{len(emit_ids)} 个发射器")


In [ ]:
# ============================================================================
# 绘图 2：时序图 —— 色块条展示（Group A）
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 5.5))

for (lid, kidx, tr, tf) in firing_events:
    tn = tf / NS
    ax.barh(lid, width=8, left=tn, height=0.7, alpha=0.8,
            color=plt.cm.tab20((lid - 1) / 16.0), edgecolor="k", linewidth=0.3)

# 标注 kick 基线
for k in range(16):
    base = kick_base_time(k) / NS
    ax.axvline(base, color="gray", ls=":", lw=0.5, alpha=0.4)
    if k % 2 == 0:
        ax.text(base, N_LASERS + 0.8, f"K{k}", fontsize=6, ha="center", color="gray")
    ax.text(base, N_LASERS + 1.8, f"2us", fontsize=5, ha="center", color="orange")

ax.set_xlabel("时间 t [ns]")
ax.set_ylabel("激光器编号")
ax.set_yticks(range(1, N_LASERS + 1))
ax.set_ylim(0.5, N_LASERS + 3)
ax.set_title("激光器发光时序图（Group A，色块 = 发光事件，灰色虚线 = kick 基线）", fontsize=12)
ax.grid(alpha=0.2, axis="x")
plt.tight_layout()
plt.savefig("crosstalk_v02_timing_blocks.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================================
# 绘图 3：每个激光器多次发光的累积 TOF 波形（相对时间 = echo_time - fire_time）
# ============================================================================
# 放一个目标在距离 D。某激光器 L 在一个 sync 内发多次光，每次发光开一个 TOF 窗。
# 窗内收到的回波（自己的 + 串扰的）按【相对时间 echo_time - 该次发光时刻】记录，
# 相对时间 → 测量距离 D_meas = rel_time·c/2。
# 把 L 的多次发光波形【叠加】：
#   · 自己的回波：每次都落在相同相对时间 2D/c → 累加增强（真值峰）
#   · 串扰回波：来自别的激光器，每次落在不同相对时间 → 散开（鬼影）
# 这正是编码抑制串扰的原理。
#
# 注意：histogram bars 因点数太少看不见，改用【阶梯折线 + 散点 + 垂直线】展示。

D_CUMUL = 150.0        # 累积 TOF 用的目标距离 [m]（可调）
res_cumul = detect_echoes_for_target(D_CUMUL)

# 相对时间范围 0~2us，步长 BIN_NS 纳秒
BIN_NS = 10                      # bin 宽 [ns] 设为 10ns ≈ 1.5m
bin_w_s = BIN_NS * 1e-9
rel_bins_s = np.arange(0, 2.0e-6 + bin_w_s/2, bin_w_s)  # 时间 bin 边界 [s]
rel_centers_s = 0.5 * (rel_bins_s[:-1] + rel_bins_s[1:])
dist_axis = rel_centers_s * C_LIGHT / 2.0                 # → 距离 [m]

demo_lasers = [1, 5, 9, 13]     # 选 4 个代表性激光器（可调）
fig, axes = plt.subplots(len(demo_lasers), 1, figsize=(15, 11), sharex=True)

for ax, det_lid in zip(axes, demo_lasers):
    echoes = [r for r in res_cumul if r[3] == det_lid]
    # 计算每个回波的相对时间 = echo_time - det_fire_time
    rel_times = np.array([(r[6] - r[5]) for r in echoes])
    emit_ids  = np.array([r[0] for r in echoes])
    rel_dists = rel_times * C_LIGHT / 2.0                # 相对时间 → 距离 [m]

    # 为每个发射激光器用不同颜色画散点 + 垂直线
    emit_set = sorted(set(emit_ids))
    for eid in emit_set:
        mask = (emit_ids == eid)
        xd = rel_dists[mask]
        color = "tab:red" if eid == det_lid else plt.cm.tab20((eid - 1) / 16.0)
        lbl = f"自己(L{eid})" if eid == det_lid else f"串扰 L{eid}"
        # 垂直线 + 散点
        ax.vlines(xd, 0, np.ones_like(xd), color=color, lw=1.8, alpha=0.7)
        ax.scatter(xd, np.ones_like(xd), s=25, color=color, marker="o", zorder=5,
                   label=lbl, edgecolor="k", linewidth=0.3)
    ax.axvline(D_CUMUL, color="k", ls=":", lw=1.2, label=f"真值 {D_CUMUL:.0f}m")
    ax.set_ylabel(f"Laser {det_lid}\n回波数")
    ax.set_ylim(0, 3)
    ax.set_xlim(0, 300)   # 0-300m = TOF 2000ns 的完整范围
    ax.legend(fontsize=6, ncol=5, loc="upper right")
    ax.grid(alpha=0.2, axis="x")

axes[-1].set_xlabel("测量距离 [m]（往返时间 = 距离×2/c，完整 TOF 2000ns = 300m）")
plt.suptitle(f"各激光器累积 TOF 波形（目标 D={D_CUMUL:.0f}m；垂直线 = 每次回波，颜色=来源激光器）\n"
             f"自己回波(红色)聚在 {D_CUMUL:.0f}m 处，串扰(其他色)散开 —— 编码抑制串扰的直观体现",
             fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("crosstalk_v02_cumulative_tof.png", dpi=110, bbox_inches="tight")
plt.show()

# ---- 统计打印 ----
print("=" * 76)
print(f"各激光器累积 TOF 统计（目标 D={D_CUMUL:.0f}m，相对时间 = echo_time - fire_time）:")
print(f"  {'Laser':>6} {'收到回波数':>10} {'自己回波':>8} {'串扰回波':>8} {'串扰来源'}")
for det_lid in range(1, N_LASERS + 1):
    echoes = [r for r in res_cumul if r[3] == det_lid]
    n_self = sum(1 for r in echoes if r[0] == det_lid)
    n_cross = sum(1 for r in echoes if r[0] != det_lid)
    cross_src = sorted(set(r[0] for r in echoes if r[0] != det_lid))
    src_str = ",".join(f"L{s}" for s in cross_src)
    print(f"  {det_lid:>6d} {len(echoes):>10d} {n_self:>8d} {n_cross:>8d} {src_str}")


In [ ]:
# ============================================================================
# 总结
# ============================================================================
print("=" * 76)
print("crosstalk_sim_v02 总结")
print("=" * 76)
print()
print(f"1. 时序来源: {EXCEL_FILE} Group A (只看 A 组)")
print(f"   - {N_LASERS} 个激光器")
print(f"   - 共 {len(firing_events)} 次发光事件（{sum(1 for fe in firing_events if fe[1]=='A')} 次 A 组）")
print(f"   - 每激光器平均发光 {len(firing_events)/N_LASERS:.1f} 次")
print()
print("2. 编码参数")
print(f"   - KICK_SPACING = {KICK_SPACING*1e6:.1f} us（两个 tick 间隔）")
print(f"   - TOF 固定 = 2000ns (0.2us)")
print(f"   - tx_trig_dly: 1ns 步长（各激光器不同）")
print(f"   - tdelay: 1/12ns 步长（按 kick 定义）")
print()
print("3. 输出文件")
print("   - crosstalk_v02_echo_16ch.png: 16 通道回波接收图")
print("   - crosstalk_v02_timing_blocks.png: 时序色块图")
print("   - crosstalk_v02_cumulative_tof.png: 累积 TOF 波形")
print()
print("4. 使用说明")
print("   - 修改 D_SHOW 看不同距离的回波图")
print("   - 修改 demo_lasers 看不同激光器的累积 TOF")
print("   - 修改 LASER_ENCODING 调整编码参数（如 tx_trig_dly）")
print("   - 修改 KICK_SPACING 和 TOF 固定值")
